In [6]:
!unzip -q /content/funko-ml-main.zip -d /content/

In [7]:
import os

print(os.listdir('/content/funko-ml-main'))

['.gitignore', 'results', 'slides', 'README.md', 'requirements.txt', 'src', 'notebooks', 'data', 'writeup']


In [8]:
%cd /content/funko-ml-main/notebooks

/content/funko-ml-main/notebooks


In [9]:
import os

print(os.path.exists('../src/evaluate.py'))

True


In [11]:
import sys
import os

# Use the absolute path to the src folder
src_path = '/content/funko-ml-main/src'

sys.path.insert(0, src_path)

print("src path:", src_path)
print("evaluate.py exists:", os.path.exists(src_path + '/evaluate.py'))

from evaluate import load_data, evaluate_model, save_results

print("All imports successful!")

src path: /content/funko-ml-main/src
evaluate.py exists: True
All imports successful!


In [12]:
d, folds = load_data()

print("X_onehot shape:", d['X_onehot'].shape)
print("X_ordinal shape:", d['X_ordinal'].shape)
print("y shape:", d['y'].shape)
print("Number of folds:", len(folds))

X_onehot shape: (359, 59)
X_ordinal shape: (359, 4)
y shape: (359,)
Number of folds: 4


In [13]:
print("Cross-validation folds:\n")

for i, (train_idx, valid_idx) in enumerate(folds, start=1):
    print(
        f"Fold {i}: "
        f"Train = {len(train_idx)}, "
        f"Validation = {len(valid_idx)}"
    )

Cross-validation folds:

Fold 1: Train = 269, Validation = 90
Fold 2: Train = 269, Validation = 90
Fold 3: Train = 269, Validation = 90
Fold 4: Train = 270, Validation = 89


In [14]:
res_linear = evaluate_model(
    lambda: SVC(
        kernel='linear',
        decision_function_shape='ovo'
    ),
    d['X_onehot'],
    d['y'],
    folds
)

print("Linear SVM evaluation completed!")

Linear SVM evaluation completed!


In [15]:
print("========== LINEAR SVM RESULTS ==========\n")

print("TRAIN")
print("Precision :", res_linear['train']['precision'])
print("Recall    :", res_linear['train']['recall'])
print("F1 Score  :", res_linear['train']['f1'])

print("\nVALIDATION")
print("Precision :", res_linear['valid']['precision'])
print("Recall    :", res_linear['valid']['recall'])
print("F1 Score  :", res_linear['valid']['f1'])

print("\nVALIDATION PRECISION BY CLASS")

for label, precision in zip(
    res_linear['labels'],
    res_linear['valid_class_precision']
):
    print(f"Class {label}: {precision}")

========== LINEAR SVM RESULTS ==========

TRAIN
Precision : 0.6246876330617559
Recall    : 0.6081749965578962
F1 Score  : 0.5762717209321879

VALIDATION
Precision : 0.40767668952309677
Recall    : 0.4819288389513109
F1 Score  : 0.44042274361488204

VALIDATION PRECISION BY CLASS
Class 1: 0.49755132850241546
Class 2: 0.4818258807588076
Class 3: 0.0
Class 4: 0.0
Class 5: 0.0


In [16]:
save_results('svm_linear', res_linear)

print("Linear SVM results saved successfully!")

Linear SVM results saved successfully!


In [17]:
res_rbf = evaluate_model(
    lambda: SVC(
        kernel='rbf',
        decision_function_shape='ovo'
    ),
    d['X_onehot'],
    d['y'],
    folds
)

print("RBF SVM evaluation completed!")

RBF SVM evaluation completed!


In [18]:
print("========== RBF SVM RESULTS ==========\n")

print("TRAIN")
print("Precision :", res_rbf['train']['precision'])
print("Recall    :", res_rbf['train']['recall'])
print("F1 Score  :", res_rbf['train']['f1'])

print("\nVALIDATION")
print("Precision :", res_rbf['valid']['precision'])
print("Recall    :", res_rbf['valid']['recall'])
print("F1 Score  :", res_rbf['valid']['f1'])

print("\nVALIDATION PRECISION BY CLASS")

for label, precision in zip(
    res_rbf['labels'],
    res_rbf['valid_class_precision']
):
    print(f"Class {label}: {precision}")

========== RBF SVM RESULTS ==========

TRAIN
Precision : 0.5481502010528172
Recall    : 0.6193274129147736
F1 Score  : 0.5651519053981088

VALIDATION
Precision : 0.4027045323956024
Recall    : 0.4820848938826467
F1 Score  : 0.43820060182570214

VALIDATION PRECISION BY CLASS
Class 1: 0.486680216802168
Class 2: 0.48061987615559043
Class 3: 0.0
Class 4: 0.0
Class 5: 0.0


In [19]:
save_results('svm_rbf', res_rbf)

print("RBF SVM results saved successfully!")

RBF SVM results saved successfully!


In [20]:
comparison = pd.DataFrame({
    'Model': [
        'Linear SVM',
        'RBF SVM'
    ],

    'Train Precision': [
        res_linear['train']['precision'],
        res_rbf['train']['precision']
    ],

    'Train Recall': [
        res_linear['train']['recall'],
        res_rbf['train']['recall']
    ],

    'Train F1': [
        res_linear['train']['f1'],
        res_rbf['train']['f1']
    ],

    'Validation Precision': [
        res_linear['valid']['precision'],
        res_rbf['valid']['precision']
    ],

    'Validation Recall': [
        res_linear['valid']['recall'],
        res_rbf['valid']['recall']
    ],

    'Validation F1': [
        res_linear['valid']['f1'],
        res_rbf['valid']['f1']
    ]
})

comparison

,Model,Train Precision,Train Recall,Train F1,Validation Precision,Validation Recall,Validation F1
0,Linear SVM,0.624688,0.608175,0.576272,0.407677,0.481929,0.440423
1,RBF SVM,0.548150,0.619327,0.565152,0.402705,0.482085,0.438201


In [21]:
comparison_rounded = comparison.copy()

numeric_columns = comparison_rounded.select_dtypes(
    include=np.number
).columns

comparison_rounded[numeric_columns] = (
    comparison_rounded[numeric_columns].round(4)
)

comparison_rounded

,Model,Train Precision,Train Recall,Train F1,Validation Precision,Validation Recall,Validation F1
0,Linear SVM,0.6247,0.6082,0.5763,0.4077,0.4819,0.4404
1,RBF SVM,0.5482,0.6193,0.5652,0.4027,0.4821,0.4382


In [22]:
comparison.to_csv(
    '../results/svm_comparison.csv',
    index=False
)

print("SVM comparison saved successfully!")

SVM comparison saved successfully!


In [23]:
import os

print("Result files:")

for file in os.listdir('../results'):
    if 'svm' in file.lower():
        print("-", file)

Result files:
- svm_rbf.csv
- svm_linear.csv
- svm_comparison.csv


In [24]:
print("=" * 60)
print("02_LINEAR_SVM NOTEBOOK COMPLETED")
print("=" * 60)

print("\nModels evaluated:")
print("1. Linear SVM")
print("2. RBF SVM")

print("\nDataset:")
print("Samples:", len(d['y']))
print("One-hot features:", d['X_onehot'].shape[1])
print("Ordinal features:", d['X_ordinal'].shape[1])
print("Cross-validation folds:", len(folds))

print("\nResult files:")
print("results/svm_linear.csv")
print("results/svm_rbf.csv")
print("results/svm_comparison.csv")

print("\nDONE!")

02_LINEAR_SVM NOTEBOOK COMPLETED

Models evaluated:
1. Linear SVM
2. RBF SVM

Dataset:
Samples: 359
One-hot features: 59
Ordinal features: 4
Cross-validation folds: 4

Result files:
results/svm_linear.csv
results/svm_rbf.csv
results/svm_comparison.csv

DONE!
